# Finance Advisor Agent (No Memory)

A personal finance advisor built with the **OpenAI Agents SDK**. It tracks spending by category and gives budget advice.

> **Note:** This notebook intentionally has **no memory / session**. Each query is fully independent, so the agent does *not* remember data from previous queries. Test query 2 references "everything I told you so far" — without memory, the agent cannot recall query 1's spending. This demonstrates the limitation that a memory-enabled version would solve.

## 1. Environment setup

In [9]:
from dotenv import load_dotenv
from agents import Agent, Runner

# Load OPENAI_API_KEY from the .env file in the project root
load_dotenv()

True

## 2. Define the Finance Advisor agent

In [10]:
finance_advisor = Agent(
    name="Finance Advisor",
    model="gpt-5-mini",
    instructions=(
        "You are a personal finance advisor. When the user reports spending, "
        "break it down by category, total it up, and give clear, practical budget "
        "advice. Be concise and specific with dollar amounts. If asked about a "
        "budget, compare spending against it and suggest concrete places to cut. "
        "If you don't have prior spending data in the current message, say so "
        "clearly rather than guessing."
    ),
)

## 3. Test query 1 — report spending

Each `Runner.run` call is independent; nothing is carried over between queries.

In [11]:
query_1 = "I spent $120 on groceries, $40 on Uber, and $60 on restaurants this week."

result_1 = await Runner.run(finance_advisor, query_1)
print(result_1.final_output)

Breakdown:
- Groceries: $120
- Uber: $40
- Restaurants: $60
Total this week: $220

Practical budget moves (concrete targets and savings):
- Groceries: target $100/week. How: meal plan, buy store brands, bulk staples. Saves $20/week → $80/month.
- Uber: target $20/week. How: use transit, walk/ride-share, combine errands. Saves $20/week → $80/month.
- Restaurants: target $30/week. How: cook 1–2 dinners at home, pack lunch. Saves $30/week → $120/month.

Projected after changes: $150/week (100 + 20 + 30) → saves $70/week = $280/month.

Quick immediate actions: plan 3 dinners for the week, pack lunch 3 days, replace 2 Uber trips with transit/walk. Want me to set a monthly budget and track it?


## 4. Test query 2 — budget question referencing earlier context

This is a **fresh, independent run**. Because there is no memory, the agent has no access to query 1's spending breakdown.

In [12]:
query_2 = "My weekly budget is $250. Based on everything I told you so far, where can I cut spending next week?"

result_2 = await Runner.run(finance_advisor, query_2)
print(result_2.final_output)

I don’t have any of your spending data from earlier messages. Please paste last week’s expenses by category so I can compare to your $250 budget.

Meanwhile, here are concrete, prioritized cuts you can make next week to ensure you hit $250:

Suggested weekly target (total $250)
- Groceries: $80
- Dining out/takeout: $25
- Transportation (gas/public transit/rideshares): $35
- Bills/subscriptions (allocated weekly): $50
- Entertainment/coffee/shopping: $10
- Savings/dedebt: $50

Fast cuts (dollar savings)
- Reduce dining out by $40 (skip 2 dinners at ~$20 each) → saves $40
- Brew coffee/pack snacks for work: save $10
- Trim groceries by $20 (meal plan, list, no impulse buys) → saves $20
- Pause one subscription or downgrade streaming: save $10–15
- Replace 1–2 rideshares with transit/walking: save $15–25

Apply one or two of the above until your projected total for next week ≤ $250. Send your actual last-week numbers and I’ll give a tailored comparison and exact cuts.
